# Notebook 09 — Correlation

**Module:** 03 — Statistics and Probability  
**Notebook:** 09 of 18  
**Prerequisites:** NB05  
**Time estimate:** 60 minutes

---
## Step 1 — Motivation

Co-expression analysis, ATAC-seq / RNA-seq joint analysis, variant effect on expression
(eQTL) — all rely on correlating two continuous variables. Choosing the wrong
correlation method, or confusing correlation with causation, leads to published errors.
This notebook makes the distinction rigorous and implementable.

---
## Step 3 — Biological Background

**Co-expression:** genes that are co-regulated often show correlated expression
across samples. High Pearson r between two genes → candidate functional relationship.

**eQTL analysis:** correlate SNP genotype (0/1/2 alleles) with gene expression level
across individuals — Spearman is appropriate because genotype is ordinal.

**Correlation vs. causation:** Gene A and Gene B may both respond to a third regulator
C (confounding). High A-B correlation does not imply A regulates B.

---
## Step 4 — Mathematical Explanation

**Pearson correlation:**
$$r = \frac{\sum_i (x_i - \bar{x})(y_i - \bar{y})}{\sqrt{\sum_i(x_i-\bar{x})^2 \cdot \sum_i(y_i-\bar{y})^2}}$$

$r \in [-1, 1]$. Measures **linear** association. Sensitive to outliers.

**Spearman correlation:** compute ranks $r_x, r_y$ of $x, y$, then Pearson on the ranks.
Equivalent to:
$$r_s = 1 - \frac{6 \sum_i d_i^2}{n(n^2-1)}, \quad d_i = r_{x_i} - r_{y_i}$$
(exact only when no ties)

Measures **monotone** association — robust to outliers and non-linearity.

**Significance:** $t = r\sqrt{(n-2)/(1-r^2)} \sim t(n-2)$ under $H_0: \rho = 0$.

---
## Step 6 — Python Implementation

In [ ]:
import numpy as np
import scipy.stats as stats
import matplotlib.pyplot as plt

In [ ]:
# Cell 6.1 — Pearson and Spearman from scratch
def pearson_r(x: np.ndarray, y: np.ndarray) -> tuple:
    """Pearson correlation and p-value."""
    x, y = np.asarray(x), np.asarray(y)
    n = len(x)
    xc, yc = x - x.mean(), y - y.mean()
    r = (xc * yc).sum() / np.sqrt((xc**2).sum() * (yc**2).sum())
    t_stat = r * np.sqrt((n-2) / (1 - r**2))
    p = 2 * stats.t.sf(abs(t_stat), df=n-2)
    return r, p

def spearman_r(x: np.ndarray, y: np.ndarray) -> tuple:
    """Spearman correlation via ranks."""
    rx = stats.rankdata(x)
    ry = stats.rankdata(y)
    return pearson_r(rx, ry)

rng = np.random.default_rng(42)

# Linear relationship
x = rng.normal(0, 1, 50)
y_linear = 2*x + rng.normal(0, 0.5, 50)

# Monotone non-linear relationship
y_nonlin = np.exp(x * 0.8) + rng.normal(0, 0.3, 50)

# With outlier
y_outlier = y_linear.copy(); y_outlier[0] = 20.0

print(f"{'Dataset':<22} {'Pearson r':>10} {'p':>10} {'Spearman r':>12} {'p':>10}")
print("-" * 68)
for label, yy in [("Linear", y_linear), ("Non-linear", y_nonlin), ("With outlier", y_outlier)]:
    pr, pp = pearson_r(x, yy)
    sr, sp_ = spearman_r(x, yy)
    print(f"  {label:<20} {pr:>10.4f} {pp:>10.4f} {sr:>12.4f} {sp_:>10.4f}")

In [ ]:
# Cell 6.2 — Spurious correlation: co-expression does not imply co-regulation
rng2 = np.random.default_rng(0)
n_samples = 100

# Confounding regulator C affects both gene A and gene B
C = rng2.normal(0, 1, n_samples)  # hidden regulator
gene_A = 2 * C + rng2.normal(0, 0.5, n_samples)
gene_B = 1.5 * C + rng2.normal(0, 0.5, n_samples)

r_AB, p_AB = pearson_r(gene_A, gene_B)
print(f"Gene A ↔ Gene B correlation: r={r_AB:.3f}, p={p_AB:.2e}")
print(f"BUT: both are driven by hidden regulator C, not by each other")
print(f"Partial correlation (controlling for C) → should be near 0")

# Partial correlation (residuals after regressing out C)
from numpy.linalg import lstsq
def partial_corr(x, y, z):
    """Correlation of x and y after partialling out z."""
    X = np.column_stack([z, np.ones(len(z))])
    res_x = x - X @ lstsq(X, x, rcond=None)[0]
    res_y = y - X @ lstsq(X, y, rcond=None)[0]
    return pearson_r(res_x, res_y)

r_partial, p_partial = partial_corr(gene_A, gene_B, C)
print(f"Partial correlation (controlling for C): r={r_partial:.4f}, p={p_partial:.4f}")

---
## Step 7 — Visualization

In [ ]:
# Cell 7.1 — Correlation comparison panel
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, (label, yy) in zip(axes, [("Linear", y_linear), ("Non-linear", y_nonlin), ("With outlier", y_outlier)]):
    pr, _ = pearson_r(x, yy)
    sr, _ = spearman_r(x, yy)
    ax.scatter(x, yy, s=20, alpha=0.7, color="steelblue")
    ax.set_xlabel("Gene A"); ax.set_ylabel("Gene B")
    ax.set_title(f"{label}\nPearson r={pr:.3f}  |  Spearman r={sr:.3f}")

# Highlight outlier
axes[2].scatter(x[0], y_outlier[0], s=80, color="tomato", zorder=5, label="Outlier")
axes[2].legend(fontsize=8)

plt.tight_layout()
plt.show()

---
## Step 8 — Exercises

1. Implement `pearson_r()` and `spearman_r()` from scratch. Verify against `scipy.stats.pearsonr` and `spearmanr`.
2. Generate two completely independent variables (n=30). What is their expected Pearson r?
   Run 10,000 such pairs and compute the distribution of |r| — at what |r| would you call it 'significant' at α=0.05?
3. When is Spearman preferred over Pearson? Give two biological scenarios.
4. The partial correlation in Cell 6.2 should be near zero. Run it 100 times with different
   random seeds. What is the distribution of the partial r values? Is it centered at zero?

---
## Quiz — Active Recall

1. Write the Pearson r formula from memory.
2. What is Spearman correlation? How is it computed?
3. High correlation does not imply causation. Give one biological example of spurious correlation.
4. What is partial correlation? Why is it useful for co-expression analysis?
5. A Pearson r of 0.3 (n=100): is it significant at α=0.05? Compute the t-statistic.

---
## Reflection

**Date completed:** ____________________

> *[Can you distinguish when to use Pearson vs. Spearman? Can you give a concrete example of correlation-causation confusion in genomics?]*

---
*Next: `10_pvalues_and_misinterpretations.ipynb`*